In [41]:
!pip install transformers datasets torchaudio sentencepiece sacremoses -q
!pip install langchain langchain_community pypdf sentence-transformers faiss-cpu -q
!pip install torch accelerate bitsandbytes -q
!pip install IndicTransToolkit -q
!pip install gTTS -q
!pip install chromadb -q

In [42]:
import torch
import torchaudio
import math
from transformers import (
    AutoModelForCTC, Wav2Vec2Processor, pipeline,
    AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
)
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from IndicTransToolkit.processor import IndicProcessor
from gtts import gTTS
import IPython.display as ipd
from google.colab import files

# **KNOWLEDGE**

In [ ]:
# Upload your PDF knowledge source
print("Please upload the PDF file you want to use as the knowledge base.")
uploaded_pdf = files.upload()
pdf_path = list(uploaded_pdf.keys())[0]



# Load and split the PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, length_function=len)
chunks = text_splitter.split_documents(documents)



# Create embeddings and store in ChromaDB
model_name_emb = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name_emb, model_kwargs={'device': 'cpu'})
persist_directory = "chroma_db"
db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=persist_directory)



print(f"✅ Knowledge base created successfully with {len(chunks)} chunks.")

Please upload the PDF file you want to use as the knowledge base.


# **HINDI ASR**

In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")



# --- 1. Load Hindi ASR Model (IndicWav2Vec) ---
print("🔄 Loading Hindi ASR model...")
asr_model_hi = AutoModelForCTC.from_pretrained("ai4bharat/indicwav2vec-hindi").to(DEVICE)
processor_hi = Wav2Vec2Processor.from_pretrained("ai4bharat/indicwav2vec-hindi")


# **TELUGU ASR**

In [ ]:
# --- 2. Load Telugu ASR Model (Lamyer's Telugu Transcription) ---
print("🔄 Loading Telugu ASR model...")
asr_pipeline_te = pipeline(
    "automatic-speech-recognition",
    model="lamyer/Telugu-transcription",
    device=0 if torch.cuda.is_available() else -1,
    chunk_length_s=30
)
asr_pipeline_te.model.config.forced_decoder_ids = asr_pipeline_te.tokenizer.get_decoder_prompt_ids(language="te", task="transcribe")


# **TRANSLATION MODELS**

In [ ]:

# --- 3. Load Translation Models (IndicTrans2) ---
print("🔄 Loading Translation models...")
ip = IndicProcessor(inference=True)



# English to Indic (Hindi/Telugu)
tokenizer_en_indic = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-en-indic-1B", trust_remote_code=True)
model_en_indic = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-en-indic-1B", trust_remote_code=True, torch_dtype=torch.float16, attn_implementation="eager").to(DEVICE)




# Indic (Hindi/Telugu) to English
tokenizer_indic_en = AutoTokenizer.from_pretrained("ai4bharat/indictrans2-indic-en-1B", trust_remote_code=True)
model_indic_en = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-indic-en-1B", trust_remote_code=True, torch_dtype=torch.float16, attn_implementation="eager").to(DEVICE)


In [ ]:
!pip install langchain_groq


# **llama-3.1 for RAG**

In [ ]:
# --- 4. Load Large Language Model for RAG (Groq) ---

print("🔄 Loading Groq LLM for RAG...")

from langchain_groq import ChatGroq

# Use your Groq API key
import os
os.environ["GROQ_API_KEY"] = "-----***********----"   # 🔑 replace with your key


llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",   # ✅ Supported & fast
    temperature=0.2,
    max_tokens=512
)


print("✅ Groq LLM loaded successfully!")


# **ASR AND TRANSLATIONS**

In [ ]:
# --- ASR Functions ---
def hindi_speech_to_text(audio_path):
    speech, sr = torchaudio.load(audio_path)


    # Resample audio to 16kHz if necessary
    if sr != 16000:
        speech = torchaudio.functional.resample(speech, sr, 16000)
    
    # Prepare audio input for the model
    inputs = processor_hi(speech.squeeze().numpy(), sampling_rate=16000, return_tensors="pt", padding=True)
    
    # Run inference to get logits
    with torch.no_grad():
        logits = asr_model_hi(inputs.input_values.to(DEVICE)).logits
    

    # Get predicted token IDs from logits
    pred_ids = torch.argmax(logits, dim=-1)
    
    # Decode tokens to text
    return processor_hi.batch_decode(pred_ids)[0]


def telugu_speech_to_text(audio_path):
    return asr_pipeline_te(audio_path)["text"]




# --- Translation Function ---
def translate_text(text, src_lang, tgt_lang):
    
    # Select appropriate tokenizer and model based on source language
    if src_lang == "eng_Latn":
        tokenizer = tokenizer_en_indic
        model = model_en_indic
    else:
        tokenizer = tokenizer_indic_en
        model = model_indic_en

    # Preprocess text for translation
    batch = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)
    
    # Tokenize and prepare inputs for the model
    inputs = tokenizer(batch, padding="longest", return_tensors="pt", return_attention_mask=True).to(DEVICE)

    # Generate translation tokens using beam search
    with torch.no_grad():
        gen_tokens = model.generate(
            **inputs,
            min_length=0,
            max_length=256,
            num_beams=5,
            use_cache=False
        )

    decoded = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
    return ip.postprocess_batch(decoded, lang=tgt_lang)[0]



# **BUILDING RAG with llama**

In [ ]:
# Create a retriever from the ChromaDB vector store, retrieving top 6 similar chunks
retriever = db.as_retriever(search_kwargs={"k": 6})


# Define the prompt template for the RAG chain, instructing the model to answer based only on context


template = """
You are an expert document analyst. Your task is to answer the user's question based *only* on the provided context.

Rules:
1. Base the answer exclusively on the CONTEXT.
2. Do not use external knowledge.
3. Keep the answer concise, max 3 sentences.
4. If no relevant info exists, reply: "I don't have enough information from the document to answer that."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
prompt = PromptTemplate.from_template(template)



# Build the RAG chain: retrieve context, pass question, apply prompt, generate with LLM, parse output
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# Function to get RAG-based answer for a given question
def gemma_rag_answer(question):
    return rag_chain.invoke(question)




# **TTS**

In [ ]:
# --- TTS Function ---
def text_to_speech(text, lang, output_file="answer.mp3"):
    tts = gTTS(text=text, lang=lang)
    tts.save(output_file)
    return output_file

# **NORMALIZATION**

In [ ]:
# --- Normalization with Llama-3.1-8B-Instant ---
llm_normalizer = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant",   # ✅ Fast model
    temperature=0.7,                # Higher temp for more natural/casual text
    max_tokens=256
)


normalize_template = PromptTemplate.from_template("""
Rewrite the following text into a more casual, conversational style
in {lang}. Avoid poetic or overly formal wording, but keep the meaning intact.

TEXT: {text}
""")

normalize_chain = normalize_template | llm_normalizer | StrOutputParser()


def normalize_speech_text(raw_text, lang="Hindi"):
    return normalize_chain.invoke({"text": raw_text, "lang": lang})


# **FULL PIPELINE**

In [ ]:
def full_pipeline(audio_file, language):
    print(f"▶️ Processing for language: {language}")

    # --- Step 1: Speech to Text ---
    if language == 'hindi':
        transcribed_text = hindi_speech_to_text(audio_file)
        src_lang, tgt_lang_code = "hin_Deva", "hi"
        norm_lang = "Hindi"
    elif language == 'telugu':
        transcribed_text = telugu_speech_to_text(audio_file)
        src_lang, tgt_lang_code = "tel_Telu", "te"
        norm_lang = "Telugu"
    else:
        print("Error: Invalid language selected. Please choose 'hindi' or 'telugu'.")
        return None, None

    print(f"📝 Transcribed Text ({language}): {transcribed_text}")



    # --- Step 2: Transcribed Text to English ---
    english_question = translate_text(transcribed_text, src_lang=src_lang, tgt_lang="eng_Latn")
    print(f"➡️ Translated Question (English): {english_question}")

    # --- Step 3: RAG model gets answer in English (Llama-3.1-8B-Instant) ---
    english_answer = gemma_rag_answer(english_question)
    print(f"🧠 RAG Answer (English): {english_answer}")

    # --- Step 4: English answer back to original language ---
    translated_answer = translate_text(english_answer, src_lang="eng_Latn", tgt_lang=src_lang)
    print(f"⬅️ Translated Answer ({language}): {translated_answer}")

    # --- Step 5: Normalize answer into casual style (Llama-3.1-8B-Instant) ---
    final_answer_text = normalize_speech_text(translated_answer, lang=norm_lang)
    print(f"✨ Normalized Answer ({language}): {final_answer_text}")

    # --- Step 6: Text to Speech in the original language ---
    audio_output_path = text_to_speech(final_answer_text, lang=tgt_lang_code)
    print(f"🔊 Generated audio at: {audio_output_path}")

    return final_answer_text, audio_output_path


# **GRADIO**

In [40]:
import gradio as gr

# 🔹 Full pipeline (Gradio-ready)
def full_pipeline(audio_file, language):
    try:
        # --- Step 1: Speech to Text ---
        if language.lower() == "hindi":
            transcribed_text = hindi_speech_to_text(audio_file)
            src_lang, tgt_lang_code, norm_lang = "hin_Deva", "hi", "Hindi"
        elif language.lower() == "telugu":
            transcribed_text = telugu_speech_to_text(audio_file)
            src_lang, tgt_lang_code, norm_lang = "tel_Telu", "te", "Telugu"
        else:
            return "❌ Invalid language", "", "", "", "", None

        # --- Step 2: Translate to English ---
        english_question = translate_text(transcribed_text, src_lang=src_lang, tgt_lang="eng_Latn")

        # --- Step 3: RAG Answer (English) ---
        english_answer = gemma_rag_answer(english_question)

        # --- Step 4: Translate Answer back to original language ---
        translated_answer = translate_text(english_answer, src_lang="eng_Latn", tgt_lang=src_lang)

        # --- Step 5: Normalize Answer into casual style ---
        normalized_answer = normalize_speech_text(translated_answer, lang=norm_lang)

        # --- Step 6: Text-to-Speech ---
        audio_output_path = text_to_speech(normalized_answer, lang=tgt_lang_code)

        # ✅ Return all outputs
        return (
            transcribed_text,   # ASR Output
            english_question,   # Translated Question
            english_answer,     # RAG Answer (English)
            translated_answer,  # Translated Answer
            normalized_answer,  # Normalized Answer
            audio_output_path   # Final Audio
        )
    except Exception as e:
        return f"❌ Error: {str(e)}", "", "", "", "", None


# 🔹 Build Gradio UI
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("## 🎙️ Multilingual Realtime AI Voice Assistant ")

    with gr.Row():
        language = gr.Dropdown(
            ["Hindi", "Telugu"],
            label="Select Language",
            value="Hindi"
        )
        audio_input = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Ask your Question")

    process_btn = gr.Button("🚀 Process")

    with gr.Row():
        asr_out = gr.Textbox(label="ASR Output (Original Language)")
        eng_q_out = gr.Textbox(label="Translated Question (English)")

    with gr.Row():
        rag_out = gr.Textbox(label="RAG Answer (English)")
        back_trans_out = gr.Textbox(label="Translated Answer (Hindi/Telugu)")

    with gr.Row():
        norm_out = gr.Textbox(label="✨ Normalized Answer (Casual Hindi/Telugu)")
        audio_out = gr.Audio(label="🔊 Final Answer (Audio)")

    # Bind function
    process_btn.click(
        fn=full_pipeline,
        inputs=[audio_input, language],
        outputs=[asr_out, eng_q_out, rag_out, back_trans_out, norm_out, audio_out]
    )


# 🔹 Launch
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b73e2f2ab67fd466f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
